# Three questions about AlphaEarth geography

This notebook is the plot layer over the saved geo-subspace runs. It does not refit models.

1. **What information is in AEF?** — how much of the 64-D residential/non-residential signal sits in a short supervised subspace.
2. **How does task-relevant geometry differ by region?** — which components keep the building label and drop metro identity.
3. **Does that geometry explain geographic transfer?** — local F1 vs cross-region F1, transfer matrices, and transfer retention.

Colors: PLS / primary = `#0072B2`, PCA / comparison = `#D55E00`.


## 1. What information is in AEF?

Leave-one-metro-out logistic regression on PCA or PLS scores. Scaler, PCA, and PLS are fit on the training metros only. Threshold stays at 0.5.

Normalized performance:

$$R(k)=\frac{F1_k}{F1_{64}}$$

PLS reaches 95% of the 64-D F1 at **k = 4**. PCA needs **k = 15**. PLS-2 already retains **0.947** of the full-representation F1.

This plot is about *recoverability* of the building-function label, not yet about geographic generalization.


### F1 retention vs dimensionality

Building-function information is concentrated in a low-dimensional supervised subspace that is not aligned with AlphaEarth's highest-variance directions.

![F1 retention vs dimensionality](../figures/paper/01_retention_vs_k.png)

## 2. How does the task-relevant geometry differ by region?

Same pixel subsample for every representation: predict building function, and separately predict metro identity. Ideal representations sit toward the top-left — high building F1, low metro accuracy.


### Pareto: building F1 vs metro accuracy

PLS-2 keeps most of the building label while metro accuracy collapses. Wider maps put geography back in.

![Pareto: building F1 vs metro accuracy](../figures/paper/02_pareto_building_vs_metro.png)

## 3. Does that geometry explain geographic transfer?

For every source metro A and target B, train on A and test on B. The diagonal uses an 80/20 within-metro split so the same pixels are not scored twice.

Cross-region F1 is the mean of the off-diagonal. Local F1 is the mean of the diagonal.

PLS-2 cross-region F1 ≈ **0.704**. PLS-9 ≈ **0.607**. AEF-64 ≈ **0.586**.


### Local vs cross-region F1 against PLS width

If local F1 rises with k while cross-region F1 peaks early and falls, the extra components are a geographic overfit — a semantic bottleneck at small k.

![Local vs cross-region F1 against PLS width](../figures/paper/03_local_vs_cross_vs_k.png)

### Transfer matrices

Rows = training metro. Columns = target metro. Mean off-diagonal: AEF-64 **0.586**, PLS-2 **0.704**, PLS-9 **0.607**.

$\Delta T = T_{\mathrm{PLS2}} - T_{\mathrm{AEF64}}$ shows which source→target pairs gain from the bottleneck.


### Transfer matrices and ΔT

Blue in ΔT means PLS-2 beats the full 64-D probe on that pair.

![Transfer matrices and ΔT](../figures/paper/04_transfer_matrices.png)

### Transfer retention

$$R_{i\rightarrow j} = \frac{T_{i\rightarrow j}}{T_{j\rightarrow j}}$$

Mean off-diagonal retention: AEF-64 **0.692**, PLS-2 **0.856**.


### Transfer-retention heatmaps

Retention is the primary generalization quantity when targets differ in difficulty.

![Transfer-retention heatmaps](../figures/paper/05_transfer_retention.png)

## Data paths

| Plot | Source |
|---|---|
| Retention vs k | `outputs/baseline_loro/20261002T0015/k_summary.csv` |
| Pareto probe | `outputs/paper_curves/20261002T1100/metro_vs_building_probe.csv` |
| Local / cross vs k | `outputs/paper_curves/20261002T1100/transfer_by_k.csv` |
| Transfer matrices | `outputs/paper_curves/20261002T1100/transfer_f1_*.csv` (fallback: `transfer_matrix/20261002T0015`) |

Figures are written under `figures/paper/` and embedded above. Re-run `scripts/build_geo_paper_plots_notebook.py` after the `paper_curves` job finishes if any panel was pending.
